# B2. Prophet Sales Forecast Notebook

> **Related Module**: [B2 Prediction Models & Decision Making](../paths/b-developers/b2-prediction-models.md)
>
> **Features**: Use Prophet for 90-day SKU sales forecasting + seasonality analysis + confidence intervals
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/b2-sales-forecast.ipynb)

In [ ]:
!pip install -q prophet pandas numpy plotly

## 1. Prepare Historical Sales Data

In [ ]:
import pandas as pd
import numpy as np
from prophet import Prophet
import plotly.graph_objects as go
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# === Option 1: Upload your CSV (requires date and sales columns) ===
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0])

# === Option 2: Generate sample data (365 days) ===
np.random.seed(42)
days = 365
dates = pd.date_range(end=datetime.now(), periods=days, freq='D')

# Base sales + weekly seasonality + yearly seasonality + trend + noise
base = 20
weekly = 5 * np.sin(2 * np.pi * np.arange(days) / 7)  # Higher on weekends
yearly = 8 * np.sin(2 * np.pi * (np.arange(days) - 60) / 365)  # Q4 peak season
trend = np.linspace(0, 5, days)  # Slow growth
noise = np.random.normal(0, 3, days)

# Prime Day and BFCM spikes
sales = (base + weekly + yearly + trend + noise).clip(3)
prime_day_idx = 180  # Around July
bfcm_idx = 300  # Around November
sales[prime_day_idx:prime_day_idx+3] *= 2.5
sales[bfcm_idx:bfcm_idx+5] *= 3.0

df = pd.DataFrame({'date': dates, 'sales': sales.round(0).astype(int)})
print(f'Historical data: {len(df)} days')
print(f'Date range: {df["date"].min().date()} to {df["date"].max().date()}')
print(f'Avg daily sales: {df["sales"].mean():.1f}')
print(f'Total sales: {df["sales"].sum():,}')

# Visualize historical data
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['date'], y=df['sales'], name='Daily Sales', line=dict(color='blue', width=1)))
fig.add_trace(go.Scatter(x=df['date'], y=df['sales'].rolling(7).mean(), name='7-Day MA', line=dict(color='red', width=2)))
fig.update_layout(title='Historical Daily Sales', xaxis_title='Date', yaxis_title='Units')
fig.show()

## 2. Prophet Model Training

In [ ]:
# Prophet requires ds (date) and y (value) columns
prophet_df = df.rename(columns={'date': 'ds', 'sales': 'y'})

# Create model
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05,  # Trend flexibility
    seasonality_prior_scale=10,
)

# Add custom holidays (Prime Day, BFCM)
prime_days = pd.DataFrame({
    'holiday': 'prime_day',
    'ds': pd.to_datetime(['2025-07-15', '2026-07-15']),
    'lower_window': 0, 'upper_window': 2
})
bfcm = pd.DataFrame({
    'holiday': 'bfcm',
    'ds': pd.to_datetime(['2025-11-28', '2026-11-27']),
    'lower_window': -1, 'upper_window': 4
})
holidays = pd.concat([prime_days, bfcm])
model = Prophet(yearly_seasonality=True, weekly_seasonality=True, holidays=holidays)

# Train
model.fit(prophet_df)
print('Model trained successfully')

## 3. 90-Day Forecast

In [ ]:
# Generate next 90 days
future = model.make_future_dataframe(periods=90)
forecast = model.predict(future)

# Visualization
fig = go.Figure()
# Historical data
fig.add_trace(go.Scatter(x=prophet_df['ds'], y=prophet_df['y'], name='Actual', mode='markers', marker=dict(size=2, color='blue')))
# Forecast
future_mask = forecast['ds'] > prophet_df['ds'].max()
fig.add_trace(go.Scatter(x=forecast[future_mask]['ds'], y=forecast[future_mask]['yhat'], name='Forecast', line=dict(color='red', width=2)))
# Confidence interval
fig.add_trace(go.Scatter(x=forecast[future_mask]['ds'], y=forecast[future_mask]['yhat_upper'], name='Upper Bound', line=dict(color='pink', width=1, dash='dash')))
fig.add_trace(go.Scatter(x=forecast[future_mask]['ds'], y=forecast[future_mask]['yhat_lower'], name='Lower Bound', line=dict(color='pink', width=1, dash='dash'), fill='tonexty'))
fig.update_layout(title='90-Day Sales Forecast', xaxis_title='Date', yaxis_title='Daily Sales')
fig.show()

# Forecast summary
future_forecast = forecast[future_mask]
print(f'\n=== 90-Day Forecast Summary ===')
print(f'Predicted total sales: {future_forecast["yhat"].sum():.0f} units')
print(f'Avg daily sales: {future_forecast["yhat"].mean():.1f} units')
print(f'Best day: {future_forecast["yhat"].max():.0f} units on {future_forecast.loc[future_forecast["yhat"].idxmax(), "ds"].strftime("%Y-%m-%d")}')
print(f'Worst day: {future_forecast["yhat"].min():.0f} units')
print(f'\n95% Confidence Interval:')
print(f'  Low scenario: {future_forecast["yhat_lower"].sum():.0f} total units')
print(f'  High scenario: {future_forecast["yhat_upper"].sum():.0f} total units')

## 4. Seasonality Analysis

In [ ]:
# Weekly seasonality
fig = model.plot_components(forecast)
import matplotlib.pyplot as plt
plt.show()

# Which day of the week has the highest sales?
weekly = forecast.groupby(forecast['ds'].dt.day_name())['weekly'].mean().reindex(
    ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday'])
print('\n=== Weekly Pattern ===')
for day, val in weekly.items():
    bar = '█' * int((val + 5) * 2)
    print(f'{day:10s}: {bar} ({val:+.1f})')

## 5. Reorder Recommendation

In [ ]:
CURRENT_INVENTORY = 850
LEAD_TIME = 45  # days
SAFETY_DAYS = 14

predicted_90d = future_forecast['yhat'].sum()
predicted_daily = future_forecast['yhat'].mean()
days_of_supply = CURRENT_INVENTORY / predicted_daily
stockout_date = datetime.now() + timedelta(days=days_of_supply)
latest_order = stockout_date - timedelta(days=LEAD_TIME)

print(f'=== Reorder Recommendation ===')
print(f'Current inventory: {CURRENT_INVENTORY} units')
print(f'Predicted daily sales: {predicted_daily:.1f} units')
print(f'Days of supply: {days_of_supply:.0f} days')
print(f'Predicted stockout: {stockout_date.strftime("%Y-%m-%d")}')
print(f'Latest order date: {latest_order.strftime("%Y-%m-%d")}')

if days_of_supply < LEAD_TIME + SAFETY_DAYS:
    reorder_qty = int((predicted_daily * (60 + SAFETY_DAYS)) - CURRENT_INVENTORY + (predicted_daily * LEAD_TIME))
    print(f'\n🔴 REORDER NOW: {reorder_qty} units')
else:
    print(f'\n🟢 No immediate reorder needed')

## 6. Export

In [ ]:
forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].to_csv('sales_forecast.csv', index=False)
print('Exported to sales_forecast.csv')